In [1]:
//%useLatestDescriptors
%use serialization, kandy

In [2]:
@Serializable
public data class JmhReport(
    val jmhVersion: String,
    val benchmark: String,
    val mode: String,
    val threads: UInt,
    val forks: UInt,
    val jvm: String,
    val jvmArgs: List<String>,
    val jdkVersion: String,
    val vmName: String,
    val vmVersion: String,
    val warmupIterations: UInt,
    val warmupTime: String,
    val warmupBatchSize: UInt,
    val measurementIterations: UInt,
    val measurementTime: String,
    val measurementBatchSize: UInt,
    val params: Map<String, String> = emptyMap(),
    val primaryMetric: PrimaryMetric,
    val secondaryMetrics: Map<String, SecondaryMetric>,
) {
    public interface Metric {
        public val score: Double
        public val scoreError: Double
        public val scoreConfidence: List<Double>
        public val scorePercentiles: Map<Double, Double>
        public val scoreUnit: String
    }

    @Serializable
    public data class PrimaryMetric(
        override val score: Double,
        override val scoreError: Double,
        override val scoreConfidence: List<Double>,
        override val scorePercentiles: Map<Double, Double>,
        override val scoreUnit: String,
        val rawDataHistogram: List<List<List<List<Double>>>>? = null,
        val rawData: List<List<Double>>? = null,
    ) : Metric

    @Serializable
    public data class SecondaryMetric(
        override val score: Double,
        override val scoreError: Double,
        override val scoreConfidence: List<Double>,
        override val scorePercentiles: Map<Double, Double>,
        override val scoreUnit: String,
        val rawData: List<List<Double>>,
    ) : Metric
}

In [3]:
import java.io.File

@OptIn(ExperimentalSerializationApi::class)
val reports = Json.decodeFromStream<List<JmhReport>>(File("data/arrayAllocation-kone_null_generic.json").inputStream())

In [35]:
import org.jetbrains.kotlinx.kandy.ir.Plot

val reportsByName = reports.groupBy { it.benchmark }.mapValues { it.value.sortedBy { it.params["size"]!!.toInt() } }
val dataByName = reportsByName.mapValues {
    val value = it.value
    mapOf(
        "size" to value.map { it.params["size"]!!.toInt().toDouble() },
        "minScore" to value.map { it.primaryMetric.rawData!!.single().min() },
        "maxScore" to value.map { it.primaryMetric.rawData!!.single().max() },
    )
}

data class PlotsAndStatistics(
    val xs: List<Double>,
    val yMins: List<Double>,
    val yMaxs: List<Double>,
    val minShift: Double,
    val maxShift: Double,
    val logLogPlotWithBorders: Plot,
    val logLogSubtractionPlotWithBorders: Plot,
)

val plots = dataByName.mapValues { (name, data) ->
    val xs = data["size"]!!.map { log2(it) }
    val yMins = data["minScore"]!!.map { log2(it) }
    val yMaxs = data["maxScore"]!!.map { log2(it) }
    val minShift = yMins.zip(xs) { y, x -> y - x }.min()
    val maxShift = yMaxs.zip(xs) { y, x -> y - x }.max()
    PlotsAndStatistics(
        xs = xs,
        yMins = yMins,
        yMaxs = yMaxs,
        minShift = minShift,
        maxShift = maxShift,
        logLogPlotWithBorders = plot {
            layout.title = name.substringAfter("dev.lounres.kone.benchmarks.collections.")
            x(xs, name = "log2(size)")
            y.axis.name = "log2(time)"
            line {
                y(xs.map { it + minShift })
                color = Color.BLUE
            }
            line {
                y(xs.map { it + maxShift })
                color = Color.BLUE
            }
            errorBars {
                yMin(yMins)
                yMax(yMaxs)
                width = 0.7
                borderLine.color = Color.RED
            }
        },
        logLogSubtractionPlotWithBorders = plot {
            layout.title = name.substringAfter("dev.lounres.kone.benchmarks.collections.")
            x(xs, name = "log2(size)")
            y.axis.name = "log2(time)"
            line {
                y(xs.map { minShift })
                color = Color.BLUE
            }
            line {
                y(xs.map { maxShift })
                color = Color.BLUE
            }
            errorBars {
                yMin(yMins.zip(xs) { y, x -> y - x })
                yMax(yMaxs.zip(xs) { y, x -> y - x })
                width = 0.7
                borderLine.color = Color.RED
            }
        }
    )
}

//plotGrid(plots.values.toList(), nCol = 2)
plots.values.single().logLogPlotWithBorders
//dataByName

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="23X5wF"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"array.ArrayAllocationBenchmarks.kone_null_generic"
},
"mapping":{
},
"data":{
"y*":[2.7584566148135146,3.7584566148135146,4.343419115534671,4.758456614813515,5.080384709700876,5.343419115534671,5.565811536871118,5.758456614813515,6.080384709700877,6.343419115534671,6.565811536871118,6.758456614813515,7.080384709700877,7.3434191155346715,7.565811536871118,7.758456614813515,8.080384709700876,8.343419115534672,8.56581153687112,8.758456614813515,9.080384709700876,9.34341911553467,9.565811536871118,9.758456614813515,10.080384709700876,10.34341911553467,10.565811536871118,10.758456614813515,11.080384709700876,11.34341911553467,11.565811536871118,11.758456614813515,12.080384709700876,12.34341911553467,12.565811536871118,12.758456614813515,13.080384709700876,13.34341911553467,13.565811536871118,13.758456614813515,14.080384709700878,14.343419115534672,14.565811536871118,14.758456614813515,15.080384709700878,15.343419115534672,15.565811536871118,15.758456614813515,16.08038470970088,16.343419115534672,16.565811536871117,16.758456614813515,17.08038470970088,17.343419115534672,17.565811536871117,17.758456614813515,18.08038470970088,18.343419115534672,18.565811536871117,18.758456614813515,19.08038470970088,19.343419115534672,19.56581153687112,19.758456614813515,20.08038470970088,20.343419115534672,20.56581153687112,20.758456614813515,21.08038470970088,21.34341911553467,21.56581153687112,21.758456614813515,22.08038470970088,22.34341911553467,22.56581153687112,22.758456614813515,23.08038470970088,23.34341911553467,23.56581153687112,23.758456614813515,24.080384709700876,24.34341911553467,24.56581153687112,24.758456614813515,25.080384709700876,25.343419115534672,25.56581153687112,25.758456614813515,26.080384709700876,26.343419115534672,26.565811536871117,26.758456614813515,27.08038470970088,27.343419115534672,27.56581153687112,27.758456614813515,28.080384709700876,28.343419115534672,28.565811536871117,28.758456614813515,29.08038470970088,29.343419115534672,29.56581153687112,29.758456614813515,30.080384709700876,30.343419115534672,30.565811536871117,30.758456614813515,31.08038470970088,31.34341911553467,31.56581153687112,31.758456614813518,32.08038470970088,32.34341911553467,32.56581153687112],
"ymin":[2.133633449135769,2.0822177261556183,2.3315018660300297,2.6213488464613324,2.9815726902476674,3.057557694556197,3.184687719122738,3.2686753540287095,3.525504311199325,3.8735582243108895,3.9201893620926835,4.160305975178112,4.421881701373129,4.645926655523249,4.8417998316457265,5.033535604892692,5.089860108554073,5.399435657776463,5.471902515736265,5.638222004398419,5.869798953761429,6.363282896560567,6.591332361693196,6.77361339050375,6.969156793347458,7.191087281836246,7.396056908508389,7.533376043933247,7.841459327198697,8.11835003669186,8.335414686397526,8.43475872476381,8.786367895981776,9.028270011913309,9.250174899391356,9.441112870043225,9.738584890620643,10.023923358430507,10.21631030497301,10.404557689638187,10.710890615422368,10.957195902503182,11.181150952282007,11.379093186822107,11.705868843860882,11.97727665714174,12.212780751009971,12.398716265097388,12.749266409959993,13.002380430620486,13.241213390374456,13.432960363040149,13.777123285095165,14.063372494204861,14.314422294758774,14.352691325287257,14.679525804116661,15.123143538607337,15.35416942505185,15.325941218366573,15.63186609393534,15.900333545999372,16.35769126180696,16.307441043254478,16.616666811224206,16.876619704537166,17.09730729777695,17.30563387544974,17.636216980772343,17.915115106919075,18.155003559037873,18.367914331753372,18.715309476

In [ ]:
val start = 2

val data = buildList {
    addAll(1 ..< (1 shl start))
    for (i in start .. 29) {
        addAll((1 shl i) ..< (1 shl (i + 1)) step (1 shl (i - start)))
    }
}

println(data.size)

plot {
    x(data.map { log2(it.toDouble()) })
    points {
        y(data.map { log2(it.toDouble()) })
    }
}